# 01. Data Cleaning

This notebook covers the extraction of historical market data for current S&P 500 constituents using Yahoo Finance. It scrapes the active universe members, standardizes ticker identifiers, executes bulk price retrieval, cleans empty tickers, and exports raw datasets in optimized formats.

## 1. Environment & Setup


### 1.1 Module Imports & Environment Configuration

Configure system paths to enable absolute imports from the project root (src), enable dynamic auto-reloading for local package development, and load core Python libraries.

In [1]:
import sys
import pandas as pd
from pathlib import Path

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

from src.data.download_data import download_prices
from src.data.download_data import validate_download
from src.data.download_data import remove_empty_tickers

### 1.2 Pipeline Global Configuration

Define parameters for historical date boundaries, sampling frequency, and price adjustment flags.

In [2]:
# ============================
# DATA EXTRACTION CONFIGURATION
# ============================

START_DATE = "2010-01-01"
END_DATE = "2024-12-31"
INTERVAL = "1d"
AUTO_ADJUST = False

## 2. Universe Definition & Ticker Normalization

### 2.1 Web Scraping S&P 500 Constituents

Scrape the current list of S&P 500 constituents from Wikipedia. Yahoo Finance uses hyphens (-) instead of dots (.) for multi-class share identifiers (e.g., BRK.B becomes BRK-B). We perform this symbol standardization before querying the API.

In [3]:
# ====================================
# S&P 500 CONSTITUENTS
# ====================================

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
sp500 = pd.read_html(url, storage_options={'User-Agent': 'Mozilla/5.0'})[0]

# Yahoo Finance uses "-" instead of "."
tickers = (
    sp500["Symbol"]
    .str.replace(".", "-", regex=False)
    .tolist()
)

print(f"Number of constituents: {len(tickers)}")

Number of constituents: 503


> Note: We begin with 503 potential ticker symbols representing the current index composition (including dual-class listings). The actual usable count depends on historical API availability and non-null trade records.

## 3. Data Download & Initial Validation


### 3.1 Bulk Price Retrieval

Download OHLCV historical time series for all standardized tickers across the defined timeframe.

In [4]:
prices = download_prices(
    tickers=tickers,
    START_DATE=START_DATE,
    END_DATE=END_DATE,
    INTERVAL=INTERVAL,
    AUTO_ADJUST=AUTO_ADJUST,
)


[**                     5%                       ]  24 of 503 completed$Q: possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1735621200")
[**********************59%***                    ]  296 of 503 completed$HONA: possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1735621200")
[**********************75%***********            ]  378 of 503 completed$FDXF: possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1735621200")
[**********************88%*****************      ]  443 of 503 completed$SNDK: possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1735621200")
[*********************100%***********************]  502 of 503 c

### 3.2 Initial Extraction Validation

Generate an initial validation report to detect missing data, empty series, or failed downloads

In [5]:
report = validate_download(prices, tickers)

DOWNLOAD SUMMARY
Requested tickers : 503
Valid downloads   : 499
Missing tickers   : 0
Empty tickers (NaN): 4

Empty tickers (100% NaN values):
  - FDXF
  - HONA
  - Q
  - SNDK


> Observation: The initial validation report identifies 4 tickers with zero valid price records (100% NaN across the panel). These unresolvable tickers need to be filtered out before further processing.

## 4. Price Matrix Cleaning & Index Standardization

### 4.1 Removing Empty Tickers & Standardizing Structure

Filter out completely unpopulated ticker columns, cast the index explicitly to DatetimeIndex, and normalize column tuples into a string-based MultiIndex structure (Price, Ticker) to avoid serialization issues.

In [6]:

prices = remove_empty_tickers(prices)

# for datetime index in the correct format, we convert the index to datetime
prices.index = pd.to_datetime(prices.index)

# tranform name in string to avoid problems with MultiIndex
prices.columns = pd.MultiIndex.from_tuples(
    [(str(c[0]), str(c[1])) for c in prices.columns],
    names=["Price", "Ticker"]
)

### 4.2 Final Download Audit

Run a final verification pass on the clean price matrix.

In [7]:
report = validate_download(prices, tickers)

DOWNLOAD SUMMARY
Requested tickers : 503
Valid downloads   : 499
Missing tickers   : 4
Empty tickers (NaN): 0

Missing tickers (not returned by API):
  - FDXF
  - HONA
  - Q
  - SNDK


> Summary: Out of the initial 503 scraped ticker symbols, valid historical market data was successfully extracted and formatted for 499 constituent companies.

## 5. Storage & Persistence

Export the scraped constituent metadata as CSV and the multi-asset price time series as a compressed Parquet file to preserve multi-level indexing and data types.

In [8]:
# upload the S&P 500 constituents and prices to CSV files

prices.to_parquet("../data/raw/sp500_prices.parquet")

sp500.to_csv("../data/raw/sp500_constituents.csv", index=False)